# MedLift-3D — Kaggle pipeline

End-to-end run on a Kaggle notebook. Built around three Kaggle facts:

* **12-hour session cap** — training uses `--max-hours 11`, checkpoints, and
  resumes automatically when you rerun the same cell in a new session.
* **No internet by default** — the projector is pure PyTorch, so nothing beyond
  Kaggle's preinstalled stack is needed. Logging is CSV on disk, not wandb.
* **`/kaggle/working` is the only writable path** — auto-detected.

Enable **GPU (T4 x2 or P100)** in Settings → Accelerator before running.

> Order matters: run the gates first. Four failure modes in this problem are
> silent — they produce plausible loss curves and wrong reconstructions.

## 0 · Setup

Point `REPO` at the code. Either add this repository as a Kaggle *dataset* /
*GitHub* source, or clone it if internet is enabled.

In [ ]:
import os, sys, subprocess, pathlib

REPO = pathlib.Path('/kaggle/input/medlift3d')     # <-- adjust
if not REPO.exists():
    REPO = pathlib.Path('/kaggle/working/medlift3d')
    if not REPO.exists():
        # Needs internet enabled; otherwise attach the repo as a dataset.
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/YOURNAME/medlift3d.git', str(REPO)], check=True)

# Kaggle input is read-only, so work from a writable copy.
WORK = pathlib.Path('/kaggle/working')
if str(REPO).startswith('/kaggle/input'):
    subprocess.run(['cp', '-r', str(REPO), str(WORK / 'medlift3d')], check=True)
    REPO = WORK / 'medlift3d'

sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

import torch
print('repo   :', REPO)
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu {i}: {p.name}  {p.total_memory/2**30:.1f} GiB')

## 1 · Gates — run these before spending any GPU time

| Gate | Asserts | Guards against |
|---|---|---|
| G1 | one canonical grid, affine never dropped | comparing volumes on different grids |
| G2 | `fp` differentiable, `bp` its exact adjoint | a projection loss with **no gradient** |
| G3 | water cylinder integral = `mu_water·2R` | HU/density confusion, scale errors |
| G4 | a *perfect* reconstruction scores perfectly | metrics that cannot detect anything |

If any gate fails, stop. Every one of these failures is silent and invalidates
every downstream number.

In [ ]:
!python scripts/run_gates.py

## 2 · Data

Synthetic phantoms by default — no download, so the pipeline is verifiable
immediately. Drop to `--shape 128 128 128` if you are tight on time; the paper
target is 256³ @ 1.5 mm.

For real data, attach a Kaggle-hosted LUNA/LIDC subset and use
`scripts/prepare_lidc.py --source dir --in /kaggle/input/<dataset>` instead.
Note that route gives volumes **without** nodule masks: usable for training the
prior, not for Dice or volume error. Only LIDC-IDRI's four-reader contours
support those.

In [ ]:
!python scripts/make_phantoms.py \
    --n-cases 60 \
    --shape 256 256 256 --spacing 1.5 1.5 1.5 \
    --views-a 16 32 --views-b 15 \
    --nodules 3 --device cuda \
    --out /kaggle/working/data/phantom

## 3 · Train the prior

~48M params at 256², batch 8 with AMP ≈ 7 GB on a T4.

**Rerun this cell in a new session to resume** — it picks up from `last.pt`
with the optimiser, scaler, step and best-val intact. `--max-hours 11` stops
cleanly before Kaggle kills the session, so no progress is ever lost to the cap.

In [ ]:
!python scripts/train_prior.py \
    --data /kaggle/working/data/phantom \
    --out  /kaggle/working/runs/prior \
    --epochs 60 --batch-size 8 --base-dim 64 \
    --amp --workers 2 --max-hours 11

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('/kaggle/working/runs/prior/log.csv')
ax = log.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(6, 3.4), grid=True)
ax.set_ylabel('eps-prediction MSE'); plt.tight_layout(); plt.show()
log.tail()

## 4 · Reconstruct

Baselines first — a learned prior has to beat them to justify its existence.
`--n-posterior 8` gives the mean reconstruction *and* the per-voxel uncertainty
map; it costs ~270 MB at 256³ and is the only thing that lets a reader tell
measured structure from structure the prior invented.

In [ ]:
DATA = '/kaggle/working/data/phantom'
RECON = '/kaggle/working/runs/recon'
PRIOR = '/kaggle/working/runs/prior/best.pt'

for method in ['fbp', 'sirt_tv', 'cgls']:
    !python scripts/reconstruct.py --data {DATA} --out {RECON} \
        --track A16 --method {method} --n-iter 60 --device cuda

!python scripts/reconstruct.py --data {DATA} --out {RECON} \
    --track A16 --method diffusion --prior {PRIOR} \
    --n-steps 50 --n-posterior 8 --slice-batch 16 --device cuda

## 5 · Evaluate

Paired metrics, because this is a paired per-patient reconstruction task with
ground truth for every case — not FID/MMD, which measure distributional
similarity for *unconditional* generation. Nodule metrics are computed inside a
dilated bounding box, and PSNR/SSIM inside the lung mask.

In [ ]:
!python scripts/evaluate.py --recon {RECON} --data {DATA}

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{RECON}/*/*.png'))[:6]:
    print(f.split('/')[-2], '·', f.split('/')[-1]); display(Image(f))

## 6 · The experiments that make the contribution

The architecture alone is not the novelty — R²-Gaussian, X-Gaussian, DOLCE and
DiffusionMBIR already occupy that ground. These three are:

1. **MDVC** — minimum detectable volume change vs view count and arc. Answers the
   Volume Doubling Time objective directly, and says where absolute volumetry
   stops being trustworthy.
2. **Hallucination audit** — insert / erase / present. Yields detection
   sensitivity and the **false-positive nodule rate**, which is the number a
   clinician cares about given the 96% LDCT false-positive rate.
3. **O5 ablation** — does the Gaussian parameterisation earn its place? A negative
   answer is a legitimate result.

In [ ]:
!python scripts/mdvc.py --data {DATA} --out /kaggle/working/runs/mdvc \
    --tracks A32 A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --gains 0.05 0.10 0.20 0.30 0.50 --limit 5 --device cuda

In [ ]:
display(Image('/kaggle/working/runs/mdvc/mdvc.png'))

In [ ]:
!python scripts/hallucination.py --data {DATA} \
    --out /kaggle/working/runs/hallucination \
    --tracks A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --limit 5 --device cuda

In [ ]:
!python scripts/ablate_roi.py --data {DATA} \
    --out /kaggle/working/runs/ablate_roi \
    --track A16 --limit 3 --iters 800 --device cuda

## 7 · Collect outputs

Kaggle persists `/kaggle/working` (~20 GB). Checkpoints are large, so keep the
one you need and drop the rest before committing the notebook.

In [ ]:
import shutil, pathlib

for p in pathlib.Path('/kaggle/working/runs/prior').glob('last.pt'):
    print('consider deleting to save space:', p, f'{p.stat().st_size/2**20:.0f} MiB')

shutil.make_archive('/kaggle/working/medlift3d_results', 'zip',
                    '/kaggle/working/runs', )
print('bundled -> /kaggle/working/medlift3d_results.zip')